### STEP 1: DATA VALIDATION & QUALITY CHECK

In [17]:
import sqlite3
import pandas as pd

# =============================
# LOAD CSV → SQLITE
# =============================
df = pd.read_csv("global_ecommerce_sales.csv")

conn = sqlite3.connect("ecommerce.db")

df.to_sql("transactions", conn, if_exists="replace", index=False)

# preview data
query = "SELECT * FROM transactions LIMIT 5;"
print(pd.read_sql(query, conn))


# ============================================================
# STEP 1.1 — TOTAL ROWS
# ============================================================
query = """
SELECT COUNT(*) AS total_rows 
FROM transactions;
"""
print("\nSTEP 1.1 — TOTAL ROWS")
print(pd.read_sql(query, conn))


# ============================================================
# STEP 1.2 — NULL CHECK
# ============================================================
query = """
SELECT
    SUM(CASE WHEN Order_ID IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN Order_Date IS NULL THEN 1 ELSE 0 END) AS null_date,
    SUM(CASE WHEN Customer_Name IS NULL THEN 1 ELSE 0 END) AS null_customer,
    SUM(CASE WHEN Product_Category IS NULL THEN 1 ELSE 0 END) AS null_category,
    SUM(CASE WHEN Total_Sales IS NULL THEN 1 ELSE 0 END) AS null_sales,
    SUM(CASE WHEN Profit IS NULL THEN 1 ELSE 0 END) AS null_profit
FROM transactions;
"""
print("\nSTEP 1.2 — NULL CHECK")
print(pd.read_sql(query, conn))


# ============================================================
# STEP 1.3 — DUPLICATE CHECK
# ============================================================
query = """
SELECT Order_ID, COUNT(*) AS count
FROM transactions
GROUP BY Order_ID
HAVING COUNT(*) > 1;
"""
print("\nSTEP 1.3 — DUPLICATE CHECK")
print(pd.read_sql(query, conn))


# ============================================================
# STEP 1.4 — DATE RANGE
# ============================================================
query = """
SELECT 
    MIN(Order_Date) AS earliest,
    MAX(Order_Date) AS latest,
    COUNT(DISTINCT Order_Date) AS unique_dates
FROM transactions;
"""
print("\nSTEP 1.4 — DATE RANGE")
print(pd.read_sql(query, conn))


# ============================================================
# STEP 1.5 — CATEGORICAL CHECK
# ============================================================
print("\nSTEP 1.5 — CATEGORICAL CHECK")

query = "SELECT Customer_Segment, COUNT(*) AS count FROM transactions GROUP BY Customer_Segment;"
print("\nCustomer Segment")
print(pd.read_sql(query, conn))

query = "SELECT Product_Category, COUNT(*) AS count FROM transactions GROUP BY Product_Category;"
print("\nProduct Category")
print(pd.read_sql(query, conn))

query = "SELECT Region, COUNT(*) AS count FROM transactions GROUP BY Region;"
print("\nRegion")
print(pd.read_sql(query, conn))

query = "SELECT Payment_Method, COUNT(*) AS count FROM transactions GROUP BY Payment_Method;"
print("\nPayment Method")
print(pd.read_sql(query, conn))


# ============================================================
# STEP 1.6 — DATA ANOMALY
# ============================================================
query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Total_Sales <= 0 THEN 1 ELSE 0 END) AS zero_or_negative_sales,
    SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) AS loss_orders,
    SUM(CASE WHEN Discount_Percent = 0 THEN 1 ELSE 0 END) AS zero_discount_orders
FROM transactions;
"""
print("\nSTEP 1.6 — DATA ANOMALY")
print(pd.read_sql(query, conn))


# ============================================================
# STEP 1.7 — VALIDASI SALES
# ============================================================
query = """
SELECT 
    Order_ID,
    Total_Sales,
    ROUND(Quantity * Unit_Price * (1 - Discount_Percent / 100.0), 2) AS calculated_sales,
    ROUND(
        Total_Sales - (Quantity * Unit_Price * (1 - Discount_Percent / 100.0)), 
        2
    ) AS discrepancy
FROM transactions
WHERE ABS(
    Total_Sales - ROUND(Quantity * Unit_Price * (1 - Discount_Percent / 100.0), 2)
) > 0.01
LIMIT 10;
"""
print("\nSTEP 1.7 — VALIDASI SALES")
print(pd.read_sql(query, conn))


    Order_ID  Order_Date   Customer_Name Customer_Segment        Country  \
0  ORD-11121  2023-01-02    Karen Suzuki        Corporate  United States   
1  ORD-11244  2023-01-02  John Johansson        Corporate          Spain   
2  ORD-10325  2023-01-03  Jessica Garcia         Consumer         Mexico   
3  ORD-10467  2023-01-03    Clara Taylor        Corporate          Italy   
4  ORD-11454  2023-01-05    Felix Thomas         Consumer          Italy   

          Region Product_Category                   Product_Name  Quantity  \
0  North America       Technology  Wireless Bluetooth Headphones         3   
1         Europe       Technology     Mechanical Gaming Keyboard         4   
2  North America  Office Supplies     Binder Clips Assorted 48pc         2   
3         Europe       Technology                Webcam HD 1080p         2   
4         Europe        Furniture        Standing Desk Converter         4   

   Unit_Price  Discount_Percent  Total_Sales  Shipping_Cost  Profit  \
0  

### STEP 2: DATA ENRICHMENT (Add Derived Columns)

In [23]:
import sqlite3
import pandas as pd

cursor = conn.cursor()

# ============================================================
# 2.1 ADD MONTH
# ============================================================
cursor.execute("ALTER TABLE transactions ADD COLUMN txn_month TEXT;")

cursor.execute("""
UPDATE transactions
SET txn_month = strftime('%Y-%m', Order_Date);
""")

# ============================================================
# 2.2 ADD YEAR
# ============================================================
cursor.execute("ALTER TABLE transactions ADD COLUMN txn_year INTEGER;")

cursor.execute("""
UPDATE transactions
SET txn_year = strftime('%Y', Order_Date);
""")

# ============================================================
# 2.3 ADD QUARTER
# ============================================================
cursor.execute("ALTER TABLE transactions ADD COLUMN txn_quarter TEXT;")

cursor.execute("""
UPDATE transactions
SET txn_quarter = 
    'Q' || ((CAST(strftime('%m', Order_Date) AS INTEGER)-1)/3 + 1)
    || '-' || strftime('%Y', Order_Date);
""")

# ============================================================
# 2.4 PROFIT MARGIN
# ============================================================
cursor.execute("ALTER TABLE transactions ADD COLUMN Profit_Margin_Pct REAL;")

cursor.execute("""
UPDATE transactions
SET Profit_Margin_Pct = 
    ROUND((Profit / Total_Sales) * 100, 2);
""")

# ============================================================
# 2.5 LOSS FLAG
# ============================================================
cursor.execute("ALTER TABLE transactions ADD COLUMN Is_Loss TEXT;")

cursor.execute("""
UPDATE transactions
SET Is_Loss = 
    CASE 
        WHEN Profit < 0 THEN 'Yes'
        ELSE 'No'
    END;
""")

# ============================================================
# 2.6 DISCOUNT TIER
# ============================================================
cursor.execute("ALTER TABLE transactions ADD COLUMN Discount_Tier TEXT;")

cursor.execute("""
UPDATE transactions
SET Discount_Tier =
    CASE
        WHEN Discount_Percent = 0 THEN '0% (No Discount)'
        WHEN Discount_Percent <= 10 THEN '1–10% (Low)'
        WHEN Discount_Percent <= 20 THEN '11–20% (Medium)'
        ELSE '21%+ (High)'
    END;
""")

# ============================================================
# 2.7 AMOUNT TIER
# ============================================================
cursor.execute("ALTER TABLE transactions ADD COLUMN Amount_Tier TEXT;")

cursor.execute("""
UPDATE transactions
SET Amount_Tier =
    CASE
        WHEN Total_Sales < 100 THEN 'Small (<$100)'
        WHEN Total_Sales < 500 THEN 'Medium ($100–$499)'
        WHEN Total_Sales < 1000 THEN 'Large ($500–$999)'
        ELSE 'Enterprise ($1000+)'
    END;
""")

# ============================================================
# SAVE CHANGES
# ============================================================
conn.commit()

# ============================================================
# PREVIEW HASIL
# ============================================================
query = "SELECT txn_month, txn_year, txn_quarter, Profit_Margin_Pct, Is_Loss, Discount_Tier, Amount_Tier FROM transactions LIMIT 5;"
print(pd.read_sql(query, conn))

  txn_month  txn_year txn_quarter  Profit_Margin_Pct Is_Loss  \
0   2023-01      2023     Q1-2023              41.88      No   
1   2023-01      2023     Q1-2023              26.68      No   
2   2023-01      2023     Q1-2023              17.18      No   
3   2023-01      2023     Q1-2023              25.03      No   
4   2023-01      2023     Q1-2023              23.95      No   

      Discount_Tier          Amount_Tier  
0  0% (No Discount)   Medium ($100–$499)  
1   11–20% (Medium)   Medium ($100–$499)  
2  0% (No Discount)        Small (<$100)  
3   11–20% (Medium)   Medium ($100–$499)  
4   11–20% (Medium)  Enterprise ($1000+)  


### STEP 3 : EXECUTIVE KPIs

In [26]:
# ============================================================
# A.1 OVERALL BUSINESS KPI
# ============================================================
query = """
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT Customer_Name) AS unique_customers,
    COUNT(DISTINCT Country) AS countries_served,
    ROUND(SUM(Total_Sales), 2) AS total_revenue,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS overall_margin_pct,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value,
    ROUND(AVG(Quantity), 1) AS avg_qty_per_order,
    ROUND(SUM(Shipping_Cost), 2) AS total_shipping_cost,
    SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) AS total_loss_orders,
    ROUND(AVG(Discount_Percent), 2) AS avg_discount_pct
FROM transactions;
"""

print("\n=== A.1 OVERALL BUSINESS KPI ===")
kpi1 = pd.read_sql(query, conn)
print(kpi1)


=== A.1 OVERALL BUSINESS KPI ===
   total_orders  unique_customers  countries_served  total_revenue  \
0          2000              1534                20      484559.34   

   total_profit  overall_margin_pct  avg_order_value  avg_qty_per_order  \
0     158872.32               32.79           242.28                3.6   

   total_shipping_cost  total_loss_orders  avg_discount_pct  
0             25804.24                272              8.57  


In [28]:
# ============================================================
# A.2 REVENUE VS PROFIT
# ============================================================
query = """
SELECT
    ROUND(SUM(Total_Sales), 2) AS total_revenue,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Shipping_Cost), 2) AS total_shipping,
    ROUND(SUM(Total_Sales) - SUM(Profit), 2) AS total_cogs_and_discount,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS profit_margin_pct
FROM transactions;
"""

print("\n=== A.2 REVENUE vs PROFIT ===")
kpi2 = pd.read_sql(query, conn)
print(kpi2)


=== A.2 REVENUE vs PROFIT ===
   total_revenue  total_profit  total_shipping  total_cogs_and_discount  \
0      484559.34     158872.32        25804.24                325687.02   

   profit_margin_pct  
0              32.79  


### STEP 4 : PRODUCT ANALYSIS

In [31]:
# ============================================================
# B.1 SALES & PROFIT BY PRODUCT CATEGORY
# ============================================================
query = """
SELECT
    Product_Category,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    ROUND(AVG(Discount_Percent), 2) AS avg_discount,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value,
    SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) AS loss_orders,
    ROUND(
        SUM(CASE WHEN Profit < 0 THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 
        2
    ) AS loss_rate_pct
FROM transactions
GROUP BY Product_Category
ORDER BY total_sales DESC;
"""

print("\n=== B.1 CATEGORY PERFORMANCE ===")
b1 = pd.read_sql(query, conn)
print(b1)


=== B.1 CATEGORY PERFORMANCE ===
         Product_Category  orders  total_sales  total_profit  margin_pct  \
0               Furniture     507    256274.68      81171.57       31.67   
1              Technology     567    139518.22      48268.65       34.60   
2  Clothing & Accessories     413     69375.63      26112.94       37.64   
3         Office Supplies     513     19390.81       3319.16       17.12   

   avg_discount  avg_order_value  loss_orders  loss_rate_pct  
0          8.66           505.47           10           1.97  
1          8.41           246.06           15           2.65  
2          8.66           167.98           23           5.57  
3          8.60            37.80          224          43.66  


In [33]:
# ============================================================
# B.2 TOP 10 PRODUCTS BY REVENUE
# ============================================================
query = """
SELECT
    Product_Name,
    Product_Category,
    COUNT(*) AS total_orders,
    SUM(Quantity) AS total_qty_sold,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    ROUND(AVG(Unit_Price), 2) AS avg_unit_price
FROM transactions
GROUP BY Product_Name, Product_Category
ORDER BY total_sales DESC
LIMIT 10;
"""

print("\n=== B.2 TOP 10 PRODUCTS ===")
b2 = pd.read_sql(query, conn)
print(b2)


=== B.2 TOP 10 PRODUCTS ===
                    Product_Name Product_Category  total_orders  \
0        Standing Desk Converter        Furniture            57   
1         Ergonomic Office Chair        Furniture            42   
2           Corner L-Shaped Desk        Furniture            39   
3           Mesh Back Task Chair        Furniture            59   
4  Wireless Bluetooth Headphones       Technology            56   
5     Mechanical Gaming Keyboard       Technology            58   
6        Filing Cabinet 3-Drawer        Furniture            45   
7         Folding Table Portable        Furniture            57   
8      Portable External SSD 1TB       Technology            55   
9               Bookshelf 5-Tier        Furniture            55   

   total_qty_sold  total_sales  total_profit  margin_pct  avg_unit_price  
0             215     46614.22      14694.35       31.52          228.45  
1             167     45405.15      15104.58       33.27          294.54  
2       

In [35]:
# ============================================================
# B.3 BOTTOM 10 PRODUCTS (LOW PROFITABILITY)
# ============================================================
query = """
SELECT
    Product_Name,
    Product_Category,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct
FROM transactions
GROUP BY Product_Name, Product_Category
HAVING orders >= 5
ORDER BY margin_pct ASC
LIMIT 10;
"""

print("\n=== B.3 LOW MARGIN PRODUCTS ===")
b3 = pd.read_sql(query, conn)
print(b3)


=== B.3 LOW MARGIN PRODUCTS ===
                     Product_Name Product_Category  orders  total_sales  \
0           Paper Clips Box 500pc  Office Supplies      45       901.89   
1        Highlighters Neon Pack 6  Office Supplies      39       942.61   
2  Sticky Notes Multicolor 6-Pack  Office Supplies      39      1072.27   
3      Binder Clips Assorted 48pc  Office Supplies      58      1924.59   
4           Ballpoint Pen Pack 12  Office Supplies      50      1760.36   
5        Whiteboard Markers Set 8  Office Supplies      59      2090.67   
6              Desk Calendar 2025  Office Supplies      59      2481.21   
7               Stapler Full-Size  Office Supplies      58      2622.15   
8              Desk Organizer Set        Furniture      49      5593.62   
9              Monitor Riser Wood        Furniture      47      8131.41   

   total_profit  margin_pct  
0        -79.95       -8.86  
1         -0.37       -0.04  
2         61.90        5.77  
3        214.49      

In [37]:
# ============================================================
# B.4 CATEGORY VS DISCOUNT IMPACT
# ============================================================
query = """
SELECT
    Product_Category,
    Discount_Tier,
    COUNT(*) AS orders,
    ROUND(AVG(Profit), 2) AS avg_profit,
    ROUND(AVG(Profit_Margin_Pct), 2) AS avg_margin_pct,
    SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) AS loss_orders
FROM transactions
GROUP BY Product_Category, Discount_Tier
ORDER BY Product_Category, Discount_Tier;
"""

print("\n=== B.4 DISCOUNT IMPACT ===")
b4 = pd.read_sql(query, conn)
print(b4)


=== B.4 DISCOUNT IMPACT ===
          Product_Category     Discount_Tier  orders  avg_profit  \
0   Clothing & Accessories  0% (No Discount)     103       81.33   
1   Clothing & Accessories   11–20% (Medium)      84       44.28   
2   Clothing & Accessories       1–10% (Low)     200       65.25   
3   Clothing & Accessories       21%+ (High)      26       37.18   
4                Furniture  0% (No Discount)     127      196.08   
5                Furniture   11–20% (Medium)     123      111.53   
6                Furniture       1–10% (Low)     234      175.53   
7                Furniture       21%+ (High)      23       64.23   
8          Office Supplies  0% (No Discount)     126        9.88   
9          Office Supplies   11–20% (Medium)     134        3.70   
10         Office Supplies       1–10% (Low)     234        6.75   
11         Office Supplies       21%+ (High)      19       -0.14   
12              Technology  0% (No Discount)     147      110.08   
13              Tec

### SECTION 5 : DISCOUNT IMPACT ANALYSIS

In [40]:
# ============================================================
# C.1 LOSS RATE BY DISCOUNT PERCENT
# ============================================================
query = """
SELECT
    Discount_Percent,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) AS loss_orders,
    ROUND(
        SUM(CASE WHEN Profit < 0 THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 
        2
    ) AS loss_rate_pct,
    ROUND(AVG(Profit), 2) AS avg_profit,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value
FROM transactions
GROUP BY Discount_Percent
ORDER BY Discount_Percent;
"""

print("\n=== C.1 LOSS RATE BY DISCOUNT ===")
c1 = pd.read_sql(query, conn)
print(c1)


=== C.1 LOSS RATE BY DISCOUNT ===
   Discount_Percent  total_orders  loss_orders  loss_rate_pct  avg_profit  \
0                 0           503           56          11.13      100.81   
1                 5           488           49          10.04       94.53   
2                10           443           59          13.32       74.34   
3                15           312           65          20.83       52.89   
4                20           169           26          15.38       55.06   
5                25            67           15          22.39       38.34   
6                30            18            2          11.11       40.46   

   avg_order_value  
0           260.37  
1           267.32  
2           232.81  
3           195.75  
4           238.05  
5           198.09  
6           301.57  


In [42]:
# ============================================================
# C.2 DISCOUNT BREAK-EVEN ANALYSIS
# ============================================================
query = """
SELECT
    Discount_Percent,
    ROUND(AVG(Profit), 2) AS avg_profit,
    ROUND(AVG(Profit_Margin_Pct), 2) AS avg_margin_pct,
    CASE
        WHEN AVG(Profit) < 0 THEN 'LOSS'
        WHEN AVG(Profit) < 30 THEN 'WARNING'
        ELSE 'HEALTHY'
    END AS profitability_status
FROM transactions
GROUP BY Discount_Percent
ORDER BY Discount_Percent;
"""

print("\n=== C.2 BREAK EVEN ANALYSIS ===")
c2 = pd.read_sql(query, conn)
print(c2)


=== C.2 BREAK EVEN ANALYSIS ===
   Discount_Percent  avg_profit  avg_margin_pct profitability_status
0                 0      100.81           26.42              HEALTHY
1                 5       94.53           24.38              HEALTHY
2                10       74.34           19.76              HEALTHY
3                15       52.89           12.49              HEALTHY
4                20       55.06            9.48              HEALTHY
5                25       38.34            6.61              HEALTHY
6                30       40.46            8.17              HEALTHY


In [44]:
# ============================================================
# C.3 CATEGORY LOSS FROM DISCOUNT
# ============================================================
query = """
SELECT
    Product_Category,
    SUM(CASE WHEN Discount_Percent > 0 AND Profit < 0 THEN 1 ELSE 0 END) AS discounted_losses,
    SUM(CASE WHEN Discount_Percent = 0 AND Profit < 0 THEN 1 ELSE 0 END) AS undiscounted_losses,
    ROUND(AVG(CASE WHEN Discount_Percent > 0 THEN Profit END), 2) AS avg_profit_with_discount,
    ROUND(AVG(CASE WHEN Discount_Percent = 0 THEN Profit END), 2) AS avg_profit_no_discount
FROM transactions
GROUP BY Product_Category
ORDER BY discounted_losses DESC;
"""

print("\n=== C.3 DISCOUNT IMPACT BY CATEGORY ===")
c3 = pd.read_sql(query, conn)
print(c3)


=== C.3 DISCOUNT IMPACT BY CATEGORY ===
         Product_Category  discounted_losses  undiscounted_losses  \
0         Office Supplies                178                   46   
1  Clothing & Accessories                 16                    7   
2              Technology                 14                    1   
3               Furniture                  8                    2   

   avg_profit_with_discount  avg_profit_no_discount  
0                      5.36                    9.88  
1                     57.21                   81.33  
2                     76.40                  110.08  
3                    148.08                  196.08  


### SECTION 6 : TIME-SERIES ANALYSIS

In [47]:
# ============================================================
# D.1 MONTHLY PERFORMANCE
# ============================================================
query = """
SELECT
    txn_month,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value
FROM transactions
GROUP BY txn_month
ORDER BY txn_month;
"""

print("\n=== D.1 MONTHLY TREND ===")
d1 = pd.read_sql(query, conn)
print(d1)


=== D.1 MONTHLY TREND ===
   txn_month  orders  total_sales  total_profit  margin_pct  avg_order_value
0    2023-01      47     11275.20       3573.01       31.69           239.90
1    2023-02      52     16943.67       5980.06       35.29           325.84
2    2023-03      51     16056.45       5718.77       35.62           314.83
3    2023-04      48     12185.00       3994.25       32.78           253.85
4    2023-05      40      8218.47       2591.86       31.54           205.46
5    2023-06      59     16191.61       5240.34       32.36           274.43
6    2023-07      59     15667.73       5195.97       33.16           265.55
7    2023-08      69     12971.07       4216.32       32.51           187.99
8    2023-09      58     11989.72       3577.00       29.83           206.72
9    2023-10      63     17794.60       5573.05       31.32           282.45
10   2023-11      46      8475.49       2469.77       29.14           184.25
11   2023-12      78     16674.44       5023.34  

In [49]:
# ============================================================
# D.2 YEARLY PERFORMANCE
# ============================================================
query = """
SELECT
    txn_year AS year,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value,
    SUM(Quantity) AS units_sold
FROM transactions
GROUP BY txn_year
ORDER BY txn_year;
"""

print("\n=== D.2 YEARLY PERFORMANCE ===")
d2 = pd.read_sql(query, conn)
print(d2)


=== D.2 YEARLY PERFORMANCE ===
   year  orders  total_sales  total_profit  margin_pct  avg_order_value  \
0  2023     670    164443.45      53153.74       32.32           245.44   
1  2024     649    155150.73      51616.11       33.27           239.06   
2  2025     681    164965.16      54102.47       32.80           242.24   

   units_sold  
0        2340  
1        2355  
2        2420  


In [51]:
# ============================================================
# D.3 YEAR-OVER-YEAR GROWTH
# ============================================================
query = """
SELECT
    curr.year,
    curr.total_sales,
    prev.total_sales AS prev_year_sales,
    ROUND((curr.total_sales - prev.total_sales) / prev.total_sales * 100, 2) AS yoy_growth_pct,
    curr.total_profit,
    ROUND((curr.total_profit - prev.total_profit) / prev.total_profit * 100, 2) AS yoy_profit_growth_pct
FROM
    (SELECT txn_year AS year,
            ROUND(SUM(Total_Sales), 2) AS total_sales,
            ROUND(SUM(Profit), 2) AS total_profit
     FROM transactions GROUP BY txn_year) curr
LEFT JOIN
    (SELECT txn_year AS year,
            ROUND(SUM(Total_Sales), 2) AS total_sales,
            ROUND(SUM(Profit), 2) AS total_profit
     FROM transactions GROUP BY txn_year) prev
ON curr.year = prev.year + 1
ORDER BY curr.year;
"""

print("\n=== D.3 YOY GROWTH ===")
d3 = pd.read_sql(query, conn)
print(d3)


=== D.3 YOY GROWTH ===
   year  total_sales  prev_year_sales  yoy_growth_pct  total_profit  \
0  2023    164443.45              NaN             NaN      53153.74   
1  2024    155150.73        164443.45           -5.65      51616.11   
2  2025    164965.16        155150.73            6.33      54102.47   

   yoy_profit_growth_pct  
0                    NaN  
1                  -2.89  
2                   4.82  


In [53]:
# ============================================================
# D.4 QUARTERLY PERFORMANCE
# ============================================================
query = """
SELECT
    txn_quarter AS quarter,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct
FROM transactions
GROUP BY txn_quarter
ORDER BY quarter;
"""

print("\n=== D.4 QUARTERLY PERFORMANCE ===")
d4 = pd.read_sql(query, conn)
print(d4)


=== D.4 QUARTERLY PERFORMANCE ===
    quarter  orders  total_sales  total_profit  margin_pct
0   Q1-2023     150     44275.32      15271.84       34.49
1   Q1-2024     128     30544.92      10204.71       33.41
2   Q1-2025     131     27430.79       9230.12       33.65
3   Q2-2023     147     36595.08      11826.45       32.32
4   Q2-2024     175     41644.41      13861.79       33.29
5   Q2-2025     201     52093.49      16936.04       32.51
6   Q3-2023     186     40628.52      12989.29       31.97
7   Q3-2024     163     34457.74      11431.15       33.17
8   Q3-2025     161     40744.02      13335.69       32.73
9   Q4-2023     187     42944.53      13066.16       30.43
10  Q4-2024     183     48503.66      16118.46       33.23
11  Q4-2025     188     44696.86      14600.62       32.67


In [55]:
# ============================================================
# D.5 MONTH-OVER-MONTH GROWTH
# ============================================================
query = """
SELECT
    txn_month,
    total_sales,
    LAG(total_sales) OVER (ORDER BY txn_month) AS prev_month_sales,
    ROUND(total_sales - LAG(total_sales) OVER (ORDER BY txn_month), 2) AS mom_change,
    ROUND(
        (total_sales - LAG(total_sales) OVER (ORDER BY txn_month)) 
        / LAG(total_sales) OVER (ORDER BY txn_month) * 100, 2
    ) AS mom_growth_pct
FROM (
    SELECT txn_month, ROUND(SUM(Total_Sales), 2) AS total_sales
    FROM transactions GROUP BY txn_month
) monthly_sales
ORDER BY txn_month;
"""

print("\n=== D.5 MOM GROWTH ===")
d5 = pd.read_sql(query, conn)
print(d5)


=== D.5 MOM GROWTH ===
   txn_month  total_sales  prev_month_sales  mom_change  mom_growth_pct
0    2023-01     11275.20               NaN         NaN             NaN
1    2023-02     16943.67          11275.20     5668.47           50.27
2    2023-03     16056.45          16943.67     -887.22           -5.24
3    2023-04     12185.00          16056.45    -3871.45          -24.11
4    2023-05      8218.47          12185.00    -3966.53          -32.55
5    2023-06     16191.61           8218.47     7973.14           97.01
6    2023-07     15667.73          16191.61     -523.88           -3.24
7    2023-08     12971.07          15667.73    -2696.66          -17.21
8    2023-09     11989.72          12971.07     -981.35           -7.57
9    2023-10     17794.60          11989.72     5804.88           48.42
10   2023-11      8475.49          17794.60    -9319.11          -52.37
11   2023-12     16674.44           8475.49     8198.95           96.74
12   2024-01      8015.21          16674

In [57]:
# ============================================================
# D.6 CUMULATIVE PERFORMANCE
# ============================================================
query = """
SELECT
    txn_month,
    monthly_sales,
    SUM(monthly_sales) OVER (ORDER BY txn_month) AS cumulative_sales,
    SUM(monthly_profit) OVER (ORDER BY txn_month) AS cumulative_profit
FROM (
    SELECT txn_month,
           ROUND(SUM(Total_Sales), 2) AS monthly_sales,
           ROUND(SUM(Profit), 2) AS monthly_profit
    FROM transactions GROUP BY txn_month
) m
ORDER BY txn_month;
"""

print("\n=== D.6 CUMULATIVE SALES ===")
d6 = pd.read_sql(query, conn)
print(d6)


=== D.6 CUMULATIVE SALES ===
   txn_month  monthly_sales  cumulative_sales  cumulative_profit
0    2023-01       11275.20          11275.20            3573.01
1    2023-02       16943.67          28218.87            9553.07
2    2023-03       16056.45          44275.32           15271.84
3    2023-04       12185.00          56460.32           19266.09
4    2023-05        8218.47          64678.79           21857.95
5    2023-06       16191.61          80870.40           27098.29
6    2023-07       15667.73          96538.13           32294.26
7    2023-08       12971.07         109509.20           36510.58
8    2023-09       11989.72         121498.92           40087.58
9    2023-10       17794.60         139293.52           45660.63
10   2023-11        8475.49         147769.01           48130.40
11   2023-12       16674.44         164443.45           53153.74
12   2024-01        8015.21         172458.66           55643.88
13   2024-02       11979.60         184438.26           5977

In [59]:
# ============================================================
# D.7 BEST MONTHS
# ============================================================
query = """
SELECT 
    txn_month, 
    ROUND(SUM(Total_Sales),2) AS total_sales,
    RANK() OVER (ORDER BY SUM(Total_Sales) DESC) AS revenue_rank
FROM transactions 
GROUP BY txn_month 
ORDER BY total_sales DESC 
LIMIT 5;
"""

print("\n=== TOP 5 MONTHS ===")
print(pd.read_sql(query, conn))


# ============================================================
# WORST MONTHS
# ============================================================
query = """
SELECT 
    txn_month, 
    ROUND(SUM(Total_Sales),2) AS total_sales,
    RANK() OVER (ORDER BY SUM(Total_Sales) ASC) AS revenue_rank
FROM transactions 
GROUP BY txn_month 
ORDER BY total_sales ASC 
LIMIT 5;
"""

print("\n=== WORST 5 MONTHS ===")
print(pd.read_sql(query, conn))


=== TOP 5 MONTHS ===
  txn_month  total_sales  revenue_rank
0   2025-06     18068.09             1
1   2023-10     17794.60             2
2   2025-05     17621.80             3
3   2024-10     17426.05             4
4   2023-02     16943.67             5

=== WORST 5 MONTHS ===
  txn_month  total_sales  revenue_rank
0   2025-02      7337.28             1
1   2025-03      7772.33             2
2   2024-01      8015.21             3
3   2023-05      8218.47             4
4   2023-11      8475.49             5


### SECTION 7 : CUSTOMER & SEGMENT ANALYSIS

In [62]:
# ============================================================
# E.1 CUSTOMER SEGMENT PERFORMANCE
# ============================================================
query = """
SELECT
    Customer_Segment,
    COUNT(*) AS orders,
    COUNT(DISTINCT Customer_Name) AS unique_customers,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value,
    ROUND(SUM(Total_Sales) / COUNT(DISTINCT Customer_Name), 2) AS revenue_per_customer
FROM transactions
GROUP BY Customer_Segment
ORDER BY total_sales DESC;
"""

print("\n=== E.1 CUSTOMER SEGMENT ===")
e1 = pd.read_sql(query, conn)
print(e1)


=== E.1 CUSTOMER SEGMENT ===
  Customer_Segment  orders  unique_customers  total_sales  total_profit  \
0         Consumer    1006               864    256287.74      87300.36   
1        Corporate     623               558    146050.39      44463.44   
2      Home Office     371               355     82221.21      27108.52   

   margin_pct  avg_order_value  revenue_per_customer  
0       34.06           254.76                296.63  
1       30.44           234.43                261.74  
2       32.97           221.62                231.61  


In [64]:
# ============================================================
# E.2 TOP CUSTOMERS
# ============================================================
query = """
SELECT
    Customer_Name,
    Customer_Segment,
    Country,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    GROUP_CONCAT(DISTINCT Product_Category) AS categories_bought
FROM transactions
GROUP BY Customer_Name, Customer_Segment, Country
ORDER BY total_sales DESC
LIMIT 15;
"""

print("\n=== E.2 TOP 15 CUSTOMERS ===")
e2 = pd.read_sql(query, conn)
print(e2)


=== E.2 TOP 15 CUSTOMERS ===
     Customer_Name Customer_Segment         Country  orders  total_sales  \
0      Hanna Jones        Corporate           China       1      3813.98   
1    Priya Jackson         Consumer           Italy       1      3783.56   
2       Sarah Chen         Consumer         Germany       1      3094.04   
3       Li Laurent         Consumer   United States       1      3007.62   
4   Priya Oliveira         Consumer  United Kingdom       1      2901.11   
5       Nadia Wang         Consumer           Spain       2      2892.39   
6   Felix Thompson         Consumer          Canada       1      2835.36   
7      Linda Weber        Corporate           Japan       1      2762.40   
8   Layla Oliveira      Home Office           Italy       1      2701.31   
9     Marco Santos         Consumer          Mexico       1      2620.28   
10      Karen Sato        Corporate          Brazil       1      2616.41   
11     Wei Jackson      Home Office  United Kingdom       

In [66]:
# ============================================================
# E.3 HIGH VALUE CUSTOMERS
# ============================================================
query = """
SELECT
    Customer_Name,
    Customer_Segment,
    COUNT(*) AS orders,
    ROUND(AVG(Total_Sales), 2) AS personal_avg,
    ROUND((SELECT AVG(Total_Sales) FROM transactions), 2) AS global_avg,
    ROUND(
        AVG(Total_Sales) - (SELECT AVG(Total_Sales) FROM transactions), 
        2
    ) AS diff_from_avg,
    ROUND(SUM(Profit), 2) AS total_profit
FROM transactions
GROUP BY Customer_Name, Customer_Segment
HAVING personal_avg > (SELECT AVG(Total_Sales) FROM transactions)
ORDER BY personal_avg DESC
LIMIT 15;
"""

print("\n=== E.3 HIGH VALUE CUSTOMERS ===")
e3 = pd.read_sql(query, conn)
print(e3)


=== E.3 HIGH VALUE CUSTOMERS ===
     Customer_Name Customer_Segment  orders  personal_avg  global_avg  \
0      Hanna Jones        Corporate       1       3813.98      242.28   
1       Li Laurent         Consumer       1       3007.62      242.28   
2   Priya Oliveira         Consumer       1       2901.11      242.28   
3   Felix Thompson         Consumer       1       2835.36      242.28   
4      Linda Weber        Corporate       1       2762.40      242.28   
5   Layla Oliveira      Home Office       1       2701.31      242.28   
6     Marco Santos         Consumer       1       2620.28      242.28   
7      Wei Jackson      Home Office       1       2566.56      242.28   
8     Clara Müller         Consumer       1       2538.50      242.28   
9     Jamal Hassan         Consumer       1       2327.47      242.28   
10   Hans Martinez         Consumer       1       2264.35      242.28   
11   Lucas Nilsson        Corporate       1       2164.33      242.28   
12        Olga Ki

In [68]:
# ============================================================
# E.4 SEGMENT vs PRODUCT CATEGORY
# ============================================================
query = """
SELECT
    Customer_Segment,
    Product_Category,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct
FROM transactions
GROUP BY Customer_Segment, Product_Category
ORDER BY Customer_Segment, total_sales DESC;
"""

print("\n=== E.4 SEGMENT x CATEGORY ===")
e4 = pd.read_sql(query, conn)
print(e4)


=== E.4 SEGMENT x CATEGORY ===
   Customer_Segment        Product_Category  orders  total_sales  margin_pct
0          Consumer               Furniture     263    140400.31       33.10
1          Consumer              Technology     283     69396.47       35.97
2          Consumer  Clothing & Accessories     216     37069.90       37.90
3          Consumer         Office Supplies     244      9421.06       19.31
4         Corporate               Furniture     156     73589.89       29.08
5         Corporate              Technology     178     46034.79       32.20
6         Corporate  Clothing & Accessories     126     20738.99       36.62
7         Corporate         Office Supplies     163      5686.72       11.31
8       Home Office               Furniture      88     42284.48       31.46
9       Home Office              Technology     106     24086.96       35.21
10      Home Office  Clothing & Accessories      71     11566.74       38.64
11      Home Office         Office Supplies 

### SECTION 8 : GEOGRAPHIC ANALYSIS

In [70]:
# ============================================================
# F.1 REVENUE BY REGION
# ============================================================
query = """
SELECT
    Region,
    COUNT(*) AS orders,
    COUNT(DISTINCT Country) AS countries,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value
FROM transactions
GROUP BY Region
ORDER BY total_sales DESC;
"""

print("\n=== F.1 REGION PERFORMANCE ===")
f1 = pd.read_sql(query, conn)
print(f1)


=== F.1 REGION PERFORMANCE ===
                 Region  orders  countries  total_sales  total_profit  \
0                Europe     503          5    137006.20      45672.16   
1         North America     578          3    133876.38      45250.09   
2          Asia Pacific     520          5    121707.51      39116.61   
3         South America     191          3     46051.13      14680.98   
4  Middle East & Africa     208          4     45918.12      14152.48   

   margin_pct  avg_order_value  
0       33.34           272.38  
1       33.80           231.62  
2       32.14           234.05  
3       31.88           241.11  
4       30.82           220.76  


In [73]:
# ============================================================
# F.2 TOP COUNTRIES BY REVENUE
# ============================================================
query = """
SELECT
    country_data.Country,
    country_data.Region,
    country_data.orders,
    country_data.total_sales,
    country_data.total_profit,
    country_data.avg_order_value,
    RANK() OVER (ORDER BY country_data.total_sales DESC) AS sales_rank
FROM (
    SELECT 
        Country, 
        Region,
        COUNT(*) AS orders,
        ROUND(SUM(Total_Sales), 2) AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit,
        ROUND(AVG(Total_Sales), 2) AS avg_order_value
    FROM transactions
    GROUP BY Country, Region
) country_data
ORDER BY sales_rank
LIMIT 10;
"""

print("\n=== F.2 TOP 10 COUNTRIES ===")
f2 = pd.read_sql(query, conn)
print(f2)


=== F.2 TOP 10 COUNTRIES ===
          Country         Region  orders  total_sales  total_profit  \
0          Mexico  North America     203     47217.30      15949.44   
1          Canada  North America     179     45326.55      15320.95   
2   United States  North America     196     41332.53      13979.70   
3           Japan   Asia Pacific     124     30950.19       9826.31   
4  United Kingdom         Europe     106     30185.46      10185.32   
5         Germany         Europe      95     28989.73       9518.07   
6           China   Asia Pacific     109     28322.85       9028.07   
7           Italy         Europe     105     27484.38       9125.06   
8          France         Europe     105     26616.81       8973.46   
9       Australia   Asia Pacific     107     25287.72       8072.37   

   avg_order_value  sales_rank  
0           232.60           1  
1           253.22           2  
2           210.88           3  
3           249.60           4  
4           284.77     

In [75]:
# ============================================================
# F.3 REGION vs PRODUCT CATEGORY
# ============================================================
query = """
SELECT
    Region,
    Product_Category,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct
FROM transactions
GROUP BY Region, Product_Category
ORDER BY Region, total_sales DESC;
"""

print("\n=== F.3 REGION x CATEGORY ===")
f3 = pd.read_sql(query, conn)
print(f3)


=== F.3 REGION x CATEGORY ===
                  Region        Product_Category  orders  total_sales  \
0           Asia Pacific               Furniture     131     63448.33   
1           Asia Pacific              Technology     140     34990.88   
2           Asia Pacific  Clothing & Accessories     101     17185.71   
3           Asia Pacific         Office Supplies     148      6082.59   
4                 Europe               Furniture     135     77277.40   
5                 Europe              Technology     146     36887.08   
6                 Europe  Clothing & Accessories     115     19138.88   
7                 Europe         Office Supplies     107      3702.84   
8   Middle East & Africa               Furniture      58     21475.34   
9   Middle East & Africa              Technology      64     15485.67   
10  Middle East & Africa  Clothing & Accessories      39      7161.71   
11  Middle East & Africa         Office Supplies      47      1795.40   
12         North Ame

In [77]:
# ============================================================
# F.4 HIGHEST AOV COUNTRIES
# ============================================================
query = """
SELECT
    Country,
    Region,
    COUNT(*) AS orders,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    RANK() OVER (ORDER BY AVG(Total_Sales) DESC) AS aov_rank
FROM transactions
GROUP BY Country, Region
HAVING orders >= 10
ORDER BY avg_order_value DESC
LIMIT 10;
"""

print("\n=== F.4 TOP AOV COUNTRIES ===")
f4 = pd.read_sql(query, conn)
print(f4)


=== F.4 TOP AOV COUNTRIES ===
          Country                Region  orders  avg_order_value  total_sales  \
0         Germany                Europe      95           305.16     28989.73   
1  United Kingdom                Europe     106           284.77     30185.46   
2          Brazil         South America      65           284.21     18473.97   
3       Argentina         South America      66           268.44     17717.00   
4           Italy                Europe     105           261.76     27484.38   
5           China          Asia Pacific     109           259.84     28322.85   
6    Saudi Arabia  Middle East & Africa      53           258.12     13680.22   
7           Spain                Europe      92           257.93     23729.82   
8          France                Europe     105           253.49     26616.81   
9          Canada         North America     179           253.22     45326.55   

   aov_rank  
0         1  
1         2  
2         3  
3         4  
4      

### SECTION 9 : PAYMENT METHOD ANALYSIS

In [81]:
# ============================================================
# G.1 PAYMENT METHOD PERFORMANCE
# ============================================================
query = """
SELECT
    Payment_Method,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_orders
FROM transactions
GROUP BY Payment_Method
ORDER BY total_sales DESC;
"""

print("\n=== G.1 PAYMENT METHOD PERFORMANCE ===")
g1 = pd.read_sql(query, conn)
print(g1)


=== G.1 PAYMENT METHOD PERFORMANCE ===
     Payment_Method  orders  total_sales  total_profit  margin_pct  \
0       Credit Card     797    192717.89      62323.24       32.34   
1            PayPal     609    143112.87      47288.80       33.04   
2  Cash on Delivery     301     77129.54      25704.89       33.33   
3     Bank Transfer     293     71599.04      23555.39       32.90   

   avg_order_value  pct_of_orders  
0           241.80          39.85  
1           235.00          30.45  
2           256.24          15.05  
3           244.37          14.65  


In [83]:
# ============================================================
# G.2 PAYMENT METHOD vs CUSTOMER SEGMENT
# ============================================================
query = """
SELECT
    Payment_Method,
    Customer_Segment,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value
FROM transactions
GROUP BY Payment_Method, Customer_Segment
ORDER BY Payment_Method, total_sales DESC;
"""

print("\n=== G.2 PAYMENT x SEGMENT ===")
g2 = pd.read_sql(query, conn)
print(g2)


=== G.2 PAYMENT x SEGMENT ===
      Payment_Method Customer_Segment  orders  total_sales  avg_order_value
0      Bank Transfer         Consumer     151     39685.84           262.82
1      Bank Transfer        Corporate      91     20933.15           230.03
2      Bank Transfer      Home Office      51     10980.05           215.30
3   Cash on Delivery         Consumer     154     45100.95           292.86
4   Cash on Delivery        Corporate      97     20302.66           209.31
5   Cash on Delivery      Home Office      50     11725.93           234.52
6        Credit Card         Consumer     384     95105.62           247.67
7        Credit Card        Corporate     260     64392.69           247.66
8        Credit Card      Home Office     153     33219.58           217.12
9             PayPal         Consumer     317     76395.33           240.99
10            PayPal        Corporate     175     40421.89           230.98
11            PayPal      Home Office     117     26295.6

### SECTION 10 : ADVANCED SQL — SQL-ONLY INSIGHTS

In [86]:
# ============================================================
# H.1 YOY GROWTH BY CATEGORY
# ============================================================
query = """
SELECT
    curr.Product_Category,
    curr.year,
    curr.total_sales,
    prev.total_sales AS prev_year_sales,
    ROUND((curr.total_sales - prev.total_sales) / prev.total_sales * 100, 2) AS yoy_growth_pct
FROM
    (SELECT txn_year AS year, Product_Category,
            ROUND(SUM(Total_Sales), 2) AS total_sales
     FROM transactions GROUP BY txn_year, Product_Category) curr
LEFT JOIN
    (SELECT txn_year AS year, Product_Category,
            ROUND(SUM(Total_Sales), 2) AS total_sales
     FROM transactions GROUP BY txn_year, Product_Category) prev
ON curr.year = prev.year + 1 
AND curr.Product_Category = prev.Product_Category
WHERE prev.total_sales IS NOT NULL
ORDER BY curr.Product_Category, curr.year;
"""

print("\n=== H.1 YOY CATEGORY ===")
h1 = pd.read_sql(query, conn)
print(h1)


=== H.1 YOY CATEGORY ===
         Product_Category  year  total_sales  prev_year_sales  yoy_growth_pct
0  Clothing & Accessories  2024     25383.08         20023.55           26.77
1  Clothing & Accessories  2025     23969.00         25383.08           -5.57
2               Furniture  2024     77555.58         92646.53          -16.29
3               Furniture  2025     86072.57         77555.58           10.98
4         Office Supplies  2024      6810.80          6048.43           12.60
5         Office Supplies  2025      6531.58          6810.80           -4.10
6              Technology  2024     45401.27         45724.94           -0.71
7              Technology  2025     48392.01         45401.27            6.59


In [88]:
# ============================================================
# H.2 PROFIT PERCENTILE
# ============================================================
query = """
SELECT
    Product_Category,
    Product_Name,
    ROUND(Profit, 2) AS profit,
    ROUND(PERCENT_RANK() OVER (PARTITION BY Product_Category ORDER BY Profit) * 100, 1) AS profit_percentile,
    CASE
        WHEN PERCENT_RANK() OVER (PARTITION BY Product_Category ORDER BY Profit) < 0.25 THEN 'Bottom 25%'
        WHEN PERCENT_RANK() OVER (PARTITION BY Product_Category ORDER BY Profit) < 0.75 THEN 'Middle 50%'
        ELSE 'Top 25%'
    END AS profit_tier
FROM transactions
ORDER BY Product_Category, profit DESC
LIMIT 30;
"""

print("\n=== H.2 PROFIT DISTRIBUTION ===")
h2 = pd.read_sql(query, conn)
print(h2)


=== H.2 PROFIT DISTRIBUTION ===
          Product_Category               Product_Name  profit  \
0   Clothing & Accessories     Business Casual Blazer  709.01   
1   Clothing & Accessories     Business Casual Blazer  475.00   
2   Clothing & Accessories     Business Casual Blazer  468.09   
3   Clothing & Accessories     Business Casual Blazer  447.81   
4   Clothing & Accessories        Travel Backpack 30L  422.90   
5   Clothing & Accessories     Business Casual Blazer  374.65   
6   Clothing & Accessories        Cotton Formal Shirt  313.26   
7   Clothing & Accessories     Business Casual Blazer  312.49   
8   Clothing & Accessories       Canvas Messenger Bag  286.92   
9   Clothing & Accessories     Business Casual Blazer  280.47   
10  Clothing & Accessories       Canvas Messenger Bag  277.41   
11  Clothing & Accessories        Travel Backpack 30L  274.69   
12  Clothing & Accessories       Polarized Sunglasses  272.54   
13  Clothing & Accessories        Cotton Formal Shirt  27

In [90]:
# ============================================================
# H.3 MONTHLY RANK
# ============================================================
query = """
SELECT
    txn_year AS year,
    txn_month AS month,
    ROUND(SUM(Total_Sales), 2) AS monthly_sales,
    RANK() OVER (PARTITION BY txn_year ORDER BY SUM(Total_Sales) DESC) AS monthly_rank_in_year
FROM transactions
GROUP BY txn_year, txn_month
ORDER BY txn_year, monthly_rank_in_year;
"""

print("\n=== H.3 MONTHLY RANK ===")
h3 = pd.read_sql(query, conn)
print(h3)


=== H.3 MONTHLY RANK ===
    year    month  monthly_sales  monthly_rank_in_year
0   2023  2023-10       17794.60                     1
1   2023  2023-02       16943.67                     2
2   2023  2023-12       16674.44                     3
3   2023  2023-06       16191.61                     4
4   2023  2023-03       16056.45                     5
5   2023  2023-07       15667.73                     6
6   2023  2023-08       12971.07                     7
7   2023  2023-04       12185.00                     8
8   2023  2023-09       11989.72                     9
9   2023  2023-01       11275.20                    10
10  2023  2023-11        8475.49                    11
11  2023  2023-05        8218.47                    12
12  2024  2024-10       17426.05                     1
13  2024  2024-05       16594.20                     2
14  2024  2024-11       16413.27                     3
15  2024  2024-04       15615.06                     4
16  2024  2024-07       14937.53       

In [92]:
# ============================================================
# H.4 MOVING AVERAGE
# ============================================================
query = """
SELECT
    txn_month,
    ROUND(SUM(Total_Sales), 2) AS monthly_sales,
    ROUND(
        AVG(SUM(Total_Sales)) OVER (
            ORDER BY txn_month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2
    ) AS moving_3m_avg
FROM transactions
GROUP BY txn_month
ORDER BY txn_month;
"""

print("\n=== H.4 MOVING AVERAGE ===")
h4 = pd.read_sql(query, conn)
print(h4)


=== H.4 MOVING AVERAGE ===
   txn_month  monthly_sales  moving_3m_avg
0    2023-01       11275.20       11275.20
1    2023-02       16943.67       14109.43
2    2023-03       16056.45       14758.44
3    2023-04       12185.00       15061.71
4    2023-05        8218.47       12153.31
5    2023-06       16191.61       12198.36
6    2023-07       15667.73       13359.27
7    2023-08       12971.07       14943.47
8    2023-09       11989.72       13542.84
9    2023-10       17794.60       14251.80
10   2023-11        8475.49       12753.27
11   2023-12       16674.44       14314.84
12   2024-01        8015.21       11055.05
13   2024-02       11979.60       12223.08
14   2024-03       10550.11       10181.64
15   2024-04       15615.06       12714.92
16   2024-05       16594.20       14253.12
17   2024-06        9435.15       13881.47
18   2024-07       14937.53       13655.63
19   2024-08        9883.83       11418.84
20   2024-09        9636.38       11485.91
21   2024-10       17426.0

In [94]:
# ============================================================
# H.5 HIGH MARGIN CUSTOMERS
# ============================================================
query = """
SELECT
    Customer_Name,
    Customer_Segment,
    COUNT(*) AS orders,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS personal_margin_pct,
    ROUND((SELECT SUM(Profit)/SUM(Total_Sales)*100 FROM transactions), 2) AS global_margin_pct
FROM transactions
GROUP BY Customer_Name, Customer_Segment
HAVING personal_margin_pct > 2 * (SELECT SUM(Profit)/SUM(Total_Sales)*100 FROM transactions)
AND COUNT(*) >= 3
ORDER BY personal_margin_pct DESC
LIMIT 10;
"""

print("\n=== H.5 HIGH MARGIN CUSTOMERS ===")
h5 = pd.read_sql(query, conn)
print(h5)


=== H.5 HIGH MARGIN CUSTOMERS ===
Empty DataFrame
Columns: [Customer_Name, Customer_Segment, orders, personal_margin_pct, global_margin_pct]
Index: []


In [96]:
# ============================================================
# H.6 CATEGORY CONTRIBUTION
# ============================================================
query = """
SELECT
    Product_Category,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Total_Sales) * 100.0 / SUM(SUM(Total_Sales)) OVER(), 2) AS revenue_share_pct,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) * 100.0 / SUM(SUM(Profit)) OVER(), 2) AS profit_share_pct
FROM transactions
GROUP BY Product_Category
ORDER BY total_sales DESC;
"""

print("\n=== H.6 CATEGORY CONTRIBUTION ===")
h6 = pd.read_sql(query, conn)
print(h6)


=== H.6 CATEGORY CONTRIBUTION ===
         Product_Category  total_sales  revenue_share_pct  total_profit  \
0               Furniture    256274.68              52.89      81171.57   
1              Technology    139518.22              28.79      48268.65   
2  Clothing & Accessories     69375.63              14.32      26112.94   
3         Office Supplies     19390.81               4.00       3319.16   

   profit_share_pct  
0             51.09  
1             30.38  
2             16.44  
3              2.09  


In [98]:
# ============================================================
# H.7 SEGMENT COMPARISON
# ============================================================
query = """
SELECT 'Consumer' AS segment, 'Revenue' AS metric, ROUND(SUM(Total_Sales),2) AS value
FROM transactions WHERE Customer_Segment='Consumer'
UNION ALL
SELECT 'Consumer', 'Profit', ROUND(SUM(Profit),2) FROM transactions WHERE Customer_Segment='Consumer'
UNION ALL
SELECT 'Corporate', 'Revenue', ROUND(SUM(Total_Sales),2) FROM transactions WHERE Customer_Segment='Corporate'
UNION ALL
SELECT 'Corporate', 'Profit', ROUND(SUM(Profit),2) FROM transactions WHERE Customer_Segment='Corporate'
UNION ALL
SELECT 'Home Office', 'Revenue', ROUND(SUM(Total_Sales),2) FROM transactions WHERE Customer_Segment='Home Office'
UNION ALL
SELECT 'Home Office', 'Profit', ROUND(SUM(Profit),2) FROM transactions WHERE Customer_Segment='Home Office';
"""

print("\n=== H.7 SEGMENT COMPARISON ===")
h7 = pd.read_sql(query, conn)
print(h7)


=== H.7 SEGMENT COMPARISON ===
       segment   metric      value
0     Consumer  Revenue  256287.74
1     Consumer   Profit   87300.36
2    Corporate  Revenue  146050.39
3    Corporate   Profit   44463.44
4  Home Office  Revenue   82221.21
5  Home Office   Profit   27108.52


In [100]:
# ============================================================
# H.8 HIGH SHIPPING COST ORDERS
# ============================================================
query = """
SELECT
    Order_ID, Customer_Name, Product_Name,
    ROUND(Total_Sales, 2) AS sale,
    ROUND(Shipping_Cost, 2) AS shipping,
    ROUND(Shipping_Cost / Total_Sales * 100, 2) AS shipping_pct,
    ROUND(Profit, 2) AS profit
FROM transactions
WHERE Shipping_Cost / Total_Sales > 0.10
ORDER BY shipping_pct DESC
LIMIT 15;
"""

print("\n=== H.8 HIGH SHIPPING COST ===")
h8 = pd.read_sql(query, conn)
print(h8)


=== H.8 HIGH SHIPPING COST ===
     Order_ID       Customer_Name                    Product_Name  sale  \
0   ORD-11001    Isabella Jackson           Paper Clips Box 500pc  2.42   
1   ORD-11269    Sophie Johansson           Paper Clips Box 500pc  4.14   
2   ORD-11430      Karen Martinez           Paper Clips Box 500pc  4.30   
3   ORD-11662   Jennifer Thompson           Paper Clips Box 500pc  2.58   
4   ORD-10407    Isabella Laurent  Sticky Notes Multicolor 6-Pack  3.42   
5   ORD-10024       Giulia Dubois           Paper Clips Box 500pc  6.62   
6   ORD-11873        Susan Garcia        Whiteboard Markers Set 8  6.24   
7   ORD-11396         Zara Moreau              Desk Calendar 2025  6.83   
8   ORD-10341      Ethan Yamamoto        Highlighters Neon Pack 6  6.44   
9   ORD-10148         Sarah Smith        Highlighters Neon Pack 6  7.02   
10  ORD-11468  Christopher Harris           Ballpoint Pen Pack 12  6.69   
11  ORD-11921         Priya Jones  Sticky Notes Multicolor 6-Pack  9

### SECTION 11 : REPORTING VIEWS

In [103]:
# ============================================================
# I.1 CATEGORY PERFORMANCE VIEW
# ============================================================
conn.execute("DROP VIEW IF EXISTS vw_category_performance;")

conn.execute("""
CREATE VIEW vw_category_performance AS
SELECT
    Product_Category,
    txn_year AS year,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) AS loss_orders
FROM transactions
GROUP BY Product_Category, txn_year;
""")

print("✅ View vw_category_performance created")

✅ View vw_category_performance created


In [105]:
# ============================================================
# I.2 SEGMENT DASHBOARD VIEW
# ============================================================
conn.execute("DROP VIEW IF EXISTS vw_segment_dashboard;")

conn.execute("""
CREATE VIEW vw_segment_dashboard AS
SELECT
    Customer_Segment,
    txn_month AS month,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(AVG(Total_Sales), 2) AS avg_order_value,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct
FROM transactions
GROUP BY Customer_Segment, txn_month;
""")

print("✅ View vw_segment_dashboard created")

✅ View vw_segment_dashboard created


In [107]:
# ============================================================
# I.3 PRODUCT RANKING VIEW
# ============================================================
conn.execute("DROP VIEW IF EXISTS vw_product_ranking;")

conn.execute("""
CREATE VIEW vw_product_ranking AS
SELECT
    Product_Name,
    Product_Category,
    COUNT(*) AS orders,
    ROUND(SUM(Total_Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(SUM(Profit) / SUM(Total_Sales) * 100, 2) AS margin_pct,
    RANK() OVER (ORDER BY SUM(Profit) DESC) AS profit_rank,
    RANK() OVER (ORDER BY SUM(Total_Sales) DESC) AS revenue_rank
FROM transactions
GROUP BY Product_Name, Product_Category;
""")

print("✅ View vw_product_ranking created")

✅ View vw_product_ranking created


In [109]:
# lihat isi view
df_view = pd.read_sql("SELECT * FROM vw_category_performance LIMIT 5;", conn)
print(df_view)

         Product_Category  year  orders  total_sales  total_profit  \
0  Clothing & Accessories  2023     131     20023.55       7019.93   
1  Clothing & Accessories  2024     151     25383.08       9630.13   
2  Clothing & Accessories  2025     131     23969.00       9462.88   
3               Furniture  2023     180     92646.53      29096.09   
4               Furniture  2024     137     77555.58      25102.69   

   margin_pct  loss_orders  
0       35.06           13  
1       37.94            6  
2       39.48            4  
3       31.41            3  
4       32.37            2  
